# BlackOut Redactor

A simple, open-source NLP side-project that delves into the concept of entity-extraction.

### What does it do?

Upload any document. The model (dslim/bert-large-NER) will scan through, detecting any sensitive entities and redact them using black bars. Now you can share any sensitive documents without worry!

DISCLAIMER: The model's not perfect, some things can still slip through. Additionally, the model can only detect entities that fall under the categories of

*   `PER` (Person)
*   `ORG` (Organization)
*   `LOC` (Location)
*   `MISC` (Miscellaneous)

Other entities such as GPE (Geopolitical entity), DATE, or CARDINAL are not recognized and censored.

### How to run

Simply connect to a hosted runtime (T4 GPU was used during my testing) and click `Run all`. For an in-depth explanation on what each code cell does, see the Markdown cell before it for more details.

# Requirements

This cell will install all the required dependencies needed to run this app. Once run, you can skip it on future run-throughs.

In [1]:
!pip install torch pymupdf transformers gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 52.8 MB/s eta 0:00:00


# Model Initialization

This cell prepares the pretrained model (`dslim/bert-large-NER`). It also defines the main function being used, `redact_pdf()`, which will take a file, extract any sensitive entities, and search the file for those entities, blacking them out when found. At the end, it will return a censored file and a download link.

In [2]:
import pymupdf
from transformers import pipeline
import os
import gradio as gr

print("Loading model...")
nlp_ner = pipeline(
    "ner",
    model="dslim/bert-large-NER",
    aggregation_strategy="simple"
)
print("Model loaded")

def redact_pdf(pdf_filepath):

  base = os.path.basename(pdf_filepath)
  name, extension = os.path.splitext(base)
  output_filename = f"{name}_redacted.pdf"
  doc = pymupdf.open(pdf_filepath)
  detected_entities = []

  for page in doc:
    text = page.get_text()
    entities = nlp_ner(text)

    for entity in entities:
      start_idx = entity['start']
      end_idx = entity['end']
      redact_this = text[start_idx:end_idx].strip()

      confidence_score = entity['score']
      if len(redact_this) < 3 or confidence_score < 0.75: continue

      detected_entities.append(f"{entity['entity_group']} ({confidence_score}): {redact_this}")

      redact_instances = page.search_for(
          redact_this,
      )
      for inst in redact_instances:
        page.add_redact_annot(inst, fill=(0, 0, 0))

    page.apply_redactions()

  doc.save(output_filename)
  doc.close()

  entities_text = "\n".join(set(detected_entities))

  # return output_filename, entities_text
  return gr.DownloadButton(label = "Download Redacted PDF", value = output_filename, interactive = True), entities_text

Loading model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.33G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: dslim/bert-large-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/40.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Model loaded


# Frontend Setup

This cell initializes the frontend as a lightweight Gradio website. For now, I am simply using Gradio to rapidly test prototypes of this application; in the future, the frontend may be built using more appropriate tools and frameworks, such as React and Node.js.

When the cell is run, simply wait for the Gradio link to pop up and click it; the frontend will open up in a new tab.

In [3]:
import gradio as gr

with gr.Blocks(theme = gr.themes.Soft()) as app:

  gr.Markdown("# BlackOut Redactor")
  gr.Markdown("Upload a PDF. Sensitive entities will be censored, allowing peace of mind when sharing important documents")

  with gr.Row():
    pdf_input = gr.File(label = "Upload PDF", file_types = [".pdf"])
    with gr.Column():
      pdf_output = gr.DownloadButton(label = "No PDF Available", interactive = False)
      entities_output = gr.Textbox("Detected entities", lines = 10)

  submit_btn = gr.Button("Redact PDF", variant = "primary")
  submit_btn.click(fn = redact_pdf, inputs = pdf_input, outputs = [pdf_output, entities_output])

app.launch(debug = False)

/tmp/ipykernel_7553/2434440204.py:3: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme = gr.themes.Soft()) as app:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://54a08afe17c020abcf.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
